# Checkpoint model embedding comparison by dataset split

This notebook reconstructs the selected checkpoint and compares surrogate and teacher protein embeddings on the inner-training, inner-validation, and outer-test splits.

The evaluation focuses on protein-specific variation rather than the large shared embedding offset. Every standardized diagnostic uses a coordinate-wise mean and standard deviation fitted once on the complete inner-training fold and then reused unchanged for all splits. The model is compared with the coordinate-wise training median where that comparison is meaningful; multivariate \(R^2\) instead uses the coordinate-wise training mean as its explicit zero-skill baseline.

The main results are presented as plots:

1. standardized cosine similarity;
2. reconstructed-vector multivariate \(R^2\) relative to the training-mean embedding;
3. standardized per-protein MAE and RMSE;
4. Spearman correlation of teacher and surrogate pairwise distances.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import subprocess
import sys
import time
from contextlib import nullcontext
from pathlib import Path
from typing import Any


def model_display_name(qualified_name: str) -> str:
    class_name = qualified_name.rsplit(".", maxsplit=1)[-1]
    return re.sub(
        r"(?<=[a-z0-9])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])",
        " ",
        class_name,
    )


def select_gpu_with_most_free_memory() -> dict[str, int | str] | None:
    visible_devices = os.environ.get("CUDA_VISIBLE_DEVICES")
    allowed_devices = (
        None
        if visible_devices is None
        else [
            token.strip()
            for token in visible_devices.split(",")
            if token.strip() not in {"", "-1"}
        ]
    )
    if allowed_devices == []:
        return None

    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=index,uuid,memory.free",
                "--format=csv,noheader,nounits",
            ],
            check=True,
            capture_output=True,
            text=True,
        )
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

    candidates = []
    for row in result.stdout.splitlines():
        index, uuid, free_memory_mib = (part.strip() for part in row.split(","))
        if allowed_devices is not None and not any(
            token == index or uuid.startswith(token) for token in allowed_devices
        ):
            continue
        candidates.append(
            {
                "index": int(index),
                "uuid": uuid,
                "free_memory_mib": int(free_memory_mib),
            }
        )

    return (
        max(candidates, key=lambda gpu: gpu["free_memory_mib"])
        if candidates
        else None
    )


SELECTED_GPU = select_gpu_with_most_free_memory()
if SELECTED_GPU is not None:
    selected_gpu_uuid = str(SELECTED_GPU["uuid"])
    if (
        "torch" in sys.modules
        and os.environ.get("CUDA_VISIBLE_DEVICES") != selected_gpu_uuid
    ):
        raise RuntimeError(
            "PyTorch is already imported. Restart the kernel so GPU selection "
            "can occur before CUDA is initialized."
        )
    os.environ["CUDA_VISIBLE_DEVICES"] = selected_gpu_uuid

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from matplotlib.patches import Patch
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr
from torch.utils.data import Subset

# Make repository imports work from either the repository root or notebooks/.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Start Jupyter in this repository or its notebooks directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from esm_surrogate.data import (
    ESMProteinDataLoader,
    ProteinEmbeddingDataset,
    load_teacher_embedding,
    restore_target_coordinates,
    transform_target_coordinates,
)
from esm_surrogate.experiment import resolve_target_normalization_mode
from mlwiz.data.splitter import Splitter
from mlwiz.data.util import load_dataset
from mlwiz.util import s2c

# Choose the experiment tree and expected model class. Set
# CONFIGURATION_OVERRIDE to inspect one config_* family, or set
# RUN_DIRECTORY_OVERRIDE to pin one exact run. If both are None, the
# lowest-loss compatible run is selected automatically.
RESULTS_ROOT = Path(os.environ.get("ESM_RESULTS_ROOT", REPO_ROOT / "RESULTS")).expanduser()
# CHECKPOINT_SEARCH_ROOT = (
#     RESULTS_ROOT / "dynamic_mlp_baseline_ProteinEmbeddingDataset"
# )
# EXPECTED_MODEL_CLASS = "esm_surrogate.baseline.DynamicMLPBaseline"
# CHECKPOINT_SEARCH_ROOT = (
#     RESULTS_ROOT / "esm_distillation_mean_centered_targets_ProteinEmbeddingDataset"
# )
CHECKPOINT_SEARCH_ROOT = (
    RESULTS_ROOT / "esm_distillation_mean_centered_targets_ProteinEmbeddingDataset"
)

EXPECTED_MODEL_CLASS = "esm_surrogate.model.AdaptiveESM"
CONFIGURATION_OVERRIDE: str | None = "config_3"
RUN_DIRECTORY_OVERRIDE: Path | None = None
CHECKPOINT_FILENAME = "last_checkpoint.pth"
METRICS_FILENAME = "metrics_data.torch"
VALIDATION_LOSS_KEY = "validation_main_loss"
SPLITS_PATH = (
    REPO_ROOT
    / "DATA_SPLITS/ProteinEmbeddingDataset/"
    "ProteinEmbeddingDataset_outer1_inner1.splits"
)

# Apply this cap independently to training, validation, and test.
# Set to None to evaluate every protein in every split.
MAX_SAMPLES_PER_SPLIT: int | None = 1000
SUBSET_SEED = 42

# Increase this for faster inference when the selected model and available GPU
# memory support larger batches.
INFERENCE_BATCH_SIZE = 4
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# Keep inference in FP32: fair-esm's rotary-position cache can retain tables
# created inside autocast, which corrupts long-sequence evaluation.
USE_BFLOAT16 = False

# Pairwise distances scale quadratically, so use a deterministic subset.
PAIRWISE_MAX_PROTEINS = 1_000
PAIRWISE_PLOT_MAX_PAIRS = 100_000
HISTOGRAM_BINS = 60

# Set to a Path to persist aligned embeddings and metrics for all three splits.
SAVE_RESULTS_TO: Path | None = None
BASELINE_CACHE_DIR = REPO_ROOT / "tmp/average_token_embedding_distribution"

SPLIT_COLORS = {
    "training": "#4C72B0",
    "validation": "#DD8452",
    "test": "#55A868",
}
MEDIAN_BASELINE_COLOR = "#C44E52"
MEDIAN_BASELINE_LABEL = "Training-set median"

if INFERENCE_BATCH_SIZE < 1:
    raise ValueError("INFERENCE_BATCH_SIZE must be positive.")
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.6g}")

## Load the selected model checkpoint, dataset, and saved splits

Unless a run directory is pinned above, the notebook compares compatible runs under the configured experiment tree using the minimum `validation_main_loss` recorded in each `metrics_data.torch` history. Set `CONFIGURATION_OVERRIDE` in the setup cell to a specific `config_*` directory name; the notebook then restricts selection to that configuration and chooses its lowest-loss compatible run. The model name used in tables and plot legends is derived from the adjacent manifest.

Training and validation come from outer fold 1 / inner fold 1; test comes from outer fold 1's held-out test indices. The overlap checks below protect against accidentally reporting leakage between these three groups. The notebook reads `target_normalization` from the checkpoint manifest, while still recognizing legacy `normalize_targets` booleans. Independently of that training choice, evaluation statistics are fitted on the complete inner training fold—not on the optionally sampled evaluation subset—and reused for all three splits.

In [ ]:
if not CHECKPOINT_SEARCH_ROOT.is_dir():
    raise FileNotFoundError(
        f"Checkpoint search directory does not exist: {CHECKPOINT_SEARCH_ROOT}"
    )
if not SPLITS_PATH.is_file():
    raise FileNotFoundError(f"Data splits do not exist: {SPLITS_PATH}")


def checkpoint_selection_record(run_directory: Path) -> dict[str, Any] | None:
    checkpoint_path = run_directory / CHECKPOINT_FILENAME
    manifest_path = run_directory / "model_manifest.json"
    metrics_path = run_directory / METRICS_FILENAME
    if not all(path.is_file() for path in (checkpoint_path, manifest_path, metrics_path)):
        return None

    with manifest_path.open(encoding="utf-8") as stream:
        candidate_manifest = json.load(stream)
    if candidate_manifest.get("model") != EXPECTED_MODEL_CLASS:
        return None

    metrics = torch.load(metrics_path, map_location="cpu", weights_only=True)
    validation_history = metrics.get("losses", {}).get(VALIDATION_LOSS_KEY)
    if not isinstance(validation_history, (list, tuple)) or not validation_history:
        raise ValueError(
            f"{metrics_path} has no non-empty losses/{VALIDATION_LOSS_KEY} history."
        )
    validation_losses = np.asarray(
        [
            float(value.item()) if isinstance(value, torch.Tensor) else float(value)
            for value in validation_history
        ],
        dtype=np.float64,
    )
    if not np.isfinite(validation_losses).all():
        raise ValueError(f"{metrics_path} contains non-finite validation losses.")
    best_history_index = int(np.argmin(validation_losses))
    configuration = next(
        (
            parent.name
            for parent in run_directory.parents
            if parent.name.startswith("config_")
        ),
        run_directory.name,
    )
    return {
        "configuration": configuration,
        "run_directory": run_directory,
        "checkpoint_path": checkpoint_path,
        "manifest_path": manifest_path,
        "validation_loss": float(validation_losses[best_history_index]),
        "validation_history_position": best_history_index + 1,
    }


if CONFIGURATION_OVERRIDE is not None and RUN_DIRECTORY_OVERRIDE is not None:
    raise ValueError("Set only one of CONFIGURATION_OVERRIDE and RUN_DIRECTORY_OVERRIDE.")

if RUN_DIRECTORY_OVERRIDE is None:
    all_selection_records = [
        record
        for checkpoint_path in CHECKPOINT_SEARCH_ROOT.rglob(CHECKPOINT_FILENAME)
        if (record := checkpoint_selection_record(checkpoint_path.parent)) is not None
    ]
    selection_records = all_selection_records
    if CONFIGURATION_OVERRIDE is not None:
        selection_records = [
            record
            for record in selection_records
            if record["configuration"] == CONFIGURATION_OVERRIDE
        ]
        if not selection_records:
            available_configurations = sorted(
                {record["configuration"] for record in all_selection_records}
            )
            raise ValueError(
                f"No compatible run found for CONFIGURATION_OVERRIDE="
                f"{CONFIGURATION_OVERRIDE!r}. Available configurations: "
                f"{available_configurations}"
            )
    if not selection_records:
        raise FileNotFoundError(
            f"No compatible run with {CHECKPOINT_FILENAME}, model_manifest.json, "
            f"and {METRICS_FILENAME} exists under {CHECKPOINT_SEARCH_ROOT}."
        )
    selected_record = min(
        selection_records,
        key=lambda record: (
            record["validation_loss"],
            str(record["checkpoint_path"]),
        ),
    )
else:
    override_run_directory = Path(RUN_DIRECTORY_OVERRIDE).expanduser()
    selected_record = checkpoint_selection_record(override_run_directory)
    if selected_record is None:
        raise FileNotFoundError(
            f"Pinned run must contain a compatible model_manifest.json, "
            f"{CHECKPOINT_FILENAME}, and {METRICS_FILENAME}: "
            f"{override_run_directory}"
        )
    selection_records = [selected_record]

RUN_DIR = selected_record["run_directory"]
CHECKPOINT_PATH = selected_record["checkpoint_path"]
SELECTED_VALIDATION_LOSS = selected_record["validation_loss"]
MANIFEST_PATH = selected_record["manifest_path"]

selection_summary = pd.DataFrame(
    [
        {
            "configuration": record["configuration"],
            VALIDATION_LOSS_KEY: record["validation_loss"],
            "selected": record is selected_record,
        }
        for record in sorted(
            selection_records, key=lambda record: record["validation_loss"]
        )
    ]
).set_index("configuration")
display(selection_summary)

with MANIFEST_PATH.open(encoding="utf-8") as stream:
    manifest = json.load(stream)
if manifest.get("model") != EXPECTED_MODEL_CLASS:
    raise ValueError(
        f"Expected {EXPECTED_MODEL_CLASS}, found {manifest.get('model')} "
        f"in {MANIFEST_PATH}."
    )
MODEL_LABEL = model_display_name(manifest["model"])
model_config = dict(manifest["config"])
target_normalization = resolve_target_normalization_mode(model_config)
checkpoint_output_space = {
    "standard": "standardized",
    "center": "mean-centered",
    None: "raw",
}[target_normalization]

storage_folder = Path(model_config["storage_folder"])
if not storage_folder.is_absolute():
    storage_folder = REPO_ROOT / storage_folder
dataset = load_dataset(str(storage_folder), ProteinEmbeddingDataset)
splitter = Splitter.load(str(SPLITS_PATH))

inner_fold = splitter.inner_folds[0][0]
outer_fold = splitter.outer_folds[0]
split_indices = {
    "training": np.asarray(inner_fold.train_idxs, dtype=np.int64),
    "validation": np.asarray(inner_fold.val_idxs, dtype=np.int64),
    "test": np.asarray(outer_fold.test_idxs, dtype=np.int64),
}

for left_name, right_name in [
    ("training", "validation"),
    ("training", "test"),
    ("validation", "test"),
]:
    overlap = np.intersect1d(
        split_indices[left_name], split_indices[right_name], assume_unique=False
    )
    if overlap.size:
        raise ValueError(
            f"{left_name} and {right_name} overlap at {overlap.size:,} indices."
        )

# Evaluation always uses training-fold statistics. The dataset transformation
# must match the coordinate system emitted by the selected checkpoint.
target_statistics_path = dataset.set_target_normalization(
    inner_fold.train_idxs,
    mode=target_normalization or "standard",
)
target_mean = dataset.target_mean.detach().to(dtype=torch.float32, device="cpu")
target_std = dataset.target_std.detach().to(dtype=torch.float32, device="cpu")
if target_normalization is None:
    dataset.clear_target_normalization()
if target_mean.shape != target_std.shape:
    raise ValueError("Target mean and standard deviation shapes do not match.")
if target_mean.numel() != dataset.dim_target:
    raise ValueError("Target statistics do not match the embedding dimension.")

display(
    pd.Series(
        {
            "model": MODEL_LABEL,
            "selected_validation_loss": SELECTED_VALIDATION_LOSS,
            "checkpoint_output_space": checkpoint_output_space,
            "evaluation_statistics": "complete inner-training fold",
            "checkpoint": str(CHECKPOINT_PATH),
            "device": str(DEVICE),
            "inference_batch_size": INFERENCE_BATCH_SIZE,
        },
        name="value",
    ).to_frame()
)

## Reconstruct and restore the checkpoint

The selected model is constructed and restored on CPU before it is moved to the evaluation device. The checkpoint's recorded validation loss is checked against the value used for automatic configuration selection.

In [ ]:
model_class = s2c(manifest["model"])
model = model_class(
    dim_input_features=manifest["dim_input_features"],
    dim_target=manifest["dim_target"],
    config=model_config,
)

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
if "model_state" not in checkpoint:
    raise KeyError(f"{CHECKPOINT_PATH} does not contain 'model_state'.")

checkpoint_epoch = int(checkpoint.get("best_epoch", checkpoint.get("epoch", -1)))
checkpoint_losses = {
    key: float(value) if isinstance(value, torch.Tensor) else value
    for key, value in checkpoint.get("losses", {}).items()
}
checkpoint_scores = {
    key: float(value) if isinstance(value, torch.Tensor) else value
    for key, value in checkpoint.get("scores", {}).items()
}
checkpoint_validation_loss = checkpoint_losses.get(VALIDATION_LOSS_KEY)
if checkpoint_validation_loss is None:
    raise KeyError(
        f"{CHECKPOINT_PATH} does not record losses/{VALIDATION_LOSS_KEY}."
    )

# THIS BREAKS IS I WANT TO USE THE LAST CHECKPOINT
# if not np.isclose(
#     checkpoint_validation_loss, SELECTED_VALIDATION_LOSS, rtol=1e-6, atol=1e-10
# ):
#     raise ValueError(
#         f"The selected checkpoint records {VALIDATION_LOSS_KEY}="
#         f"{checkpoint_validation_loss:.8g}, but its metrics history has a lower "
#         f"value of {SELECTED_VALIDATION_LOSS:.8g}. Configure training to save its "
#         "best checkpoint using validation loss before using automatic selection."
#     )
model.load_state_dict(checkpoint["model_state"])
del checkpoint

model.to(DEVICE)
model.eval()

model_summary: dict[str, Any] = {
    "epoch": checkpoint_epoch + 1,
    "parameters": sum(parameter.numel() for parameter in model.parameters()),
    VALIDATION_LOSS_KEY: checkpoint_validation_loss,
}
if hasattr(model, "instantiated_widths"):
    widths = np.asarray(model.instantiated_widths())
    model_summary["FFN_width_min/median/max"] = (
        f"{widths.min():g} / {np.median(widths):g} / {widths.max():g}"
    )
display(pd.Series(model_summary, name="value").to_frame())

## Select evaluation indices and fit the median baseline

Sampling is deterministic and independent for each split. The table reports both the complete split size and the number actually evaluated, so subset results cannot be mistaken for full-split results. The constant baseline is fitted separately from evaluation by loading every raw target in the complete inner training fold and taking its coordinate-wise median. That vector is cached under the git-ignored `tmp/` directory using the dataset identity, teacher layer, embedding dimension, and exact training indices, so subsequent runs skip the expensive target scan.

In [ ]:
def select_indices(
    available_indices: np.ndarray,
    maximum_samples: int | None,
    seed: int,
) -> np.ndarray:
    if maximum_samples is None:
        return np.sort(available_indices.copy())
    if maximum_samples < 1:
        raise ValueError("MAX_SAMPLES_PER_SPLIT must be positive or None.")
    if maximum_samples >= available_indices.size:
        return np.sort(available_indices.copy())

    rng = np.random.default_rng(seed)
    return np.sort(
        rng.choice(available_indices, size=maximum_samples, replace=False)
    )


selected_indices = {
    name: select_indices(indices, MAX_SAMPLES_PER_SPLIT, SUBSET_SEED + offset)
    for offset, (name, indices) in enumerate(split_indices.items())
}

selection_summary = pd.DataFrame(
    [
        {
            "split": name,
            "available_proteins": split_indices[name].size,
            "evaluated_proteins": indices.size,
            "coverage": indices.size / split_indices[name].size,
        }
        for offset, (name, indices) in enumerate(selected_indices.items())
    ]
).set_index("split")
display(selection_summary)


def fit_coordinatewise_median(training_indices: list[int]) -> torch.Tensor:
    if not training_indices:
        raise ValueError("The median baseline requires training targets.")

    training_targets = np.empty(
        (len(training_indices), dataset.dim_target), dtype=np.float32
    )
    for row, index in enumerate(training_indices):
        record = dataset.dataset[int(index)]
        target = load_teacher_embedding(
            record["teacher_path"],
            teacher_layer=dataset.teacher_layer,
            expected_label=record["label"],
        )
        if target.numel() != dataset.dim_target:
            raise ValueError(
                f"Training target at dataset index {index} has "
                f"{target.numel()} dimensions; expected {dataset.dim_target}."
            )
        if not torch.isfinite(target).all():
            raise ValueError(
                f"Training target at dataset index {index} is non-finite."
            )
        training_targets[row] = target.numpy()
        if (row + 1) % 5_000 == 0 or row + 1 == len(training_indices):
            print(
                f"Fitting median baseline: loaded {row + 1:,}/"
                f"{len(training_indices):,} training targets",
                end="\r" if row + 1 < len(training_indices) else "\n",
            )

    median_vector = np.median(
        training_targets, axis=0, overwrite_input=True
    ).astype(np.float32, copy=False)
    return torch.from_numpy(median_vector.copy())


def load_or_fit_coordinatewise_median(
    training_indices: list[int],
) -> tuple[torch.Tensor, Path, bool]:
    index_array = np.asarray(training_indices, dtype="<i8")
    cache_identity = {
        "version": 1,
        "fasta_path": dataset.fasta_path,
        "teacher_embeddings_dir": dataset.teacher_embeddings_dir,
        "teacher_layer": str(dataset.teacher_layer),
        "dim_target": dataset.dim_target,
    }
    digest = hashlib.sha256(
        json.dumps(cache_identity, sort_keys=True).encode("utf-8")
    )
    digest.update(index_array.tobytes())
    cache_path = BASELINE_CACHE_DIR / f"median_{digest.hexdigest()}.pt"
    expected_indices = torch.from_numpy(index_array.copy())

    if cache_path.is_file():
        cached = torch.load(cache_path, map_location="cpu", weights_only=True)
        if not isinstance(cached, dict):
            raise ValueError(f"Invalid median-baseline cache: {cache_path}")
        cached_median = cached.get("median_vector")
        cached_indices = cached.get("training_indices")
        cache_is_valid = (
            cached.get("identity") == cache_identity
            and isinstance(cached_median, torch.Tensor)
            and cached_median.shape == (dataset.dim_target,)
            and cached_median.dtype == torch.float32
            and torch.isfinite(cached_median).all()
            and isinstance(cached_indices, torch.Tensor)
            and torch.equal(cached_indices, expected_indices)
        )
        if not cache_is_valid:
            raise ValueError(f"Invalid median-baseline cache: {cache_path}")
        print(f"Loaded median baseline from {cache_path}")
        return cached_median.clone(), cache_path, True

    median_vector = fit_coordinatewise_median(training_indices)
    BASELINE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    temporary_path = cache_path.with_name(
        f".{cache_path.name}.{os.getpid()}.tmp"
    )
    try:
        torch.save(
            {
                "identity": cache_identity,
                "training_indices": expected_indices,
                "median_vector": median_vector,
            },
            temporary_path,
        )
        os.replace(temporary_path, cache_path)
    finally:
        temporary_path.unlink(missing_ok=True)
    print(f"Cached median baseline at {cache_path}")
    return median_vector, cache_path, False


(
    median_baseline_vector,
    median_baseline_cache_path,
    median_baseline_cache_hit,
) = load_or_fit_coordinatewise_median(inner_fold.train_idxs)
print(
    f"{MEDIAN_BASELINE_LABEL}: fitted on {len(inner_fold.train_idxs):,} proteins; "
    f"{median_baseline_vector.numel():,} dimensions; "
    f"cache {'hit' if median_baseline_cache_hit else 'created'}."
)

## Evaluate aligned embeddings

The function below keeps checkpoint output space and evaluation space separate:

- a standard-normalized checkpoint emits standardized embeddings directly;
- a mean-centered checkpoint emits embeddings with the training centroid removed;
- an unnormalized checkpoint emits raw embeddings;
- every checkpoint output is restored to raw coordinates, then standardized for evaluation;
- raw embeddings remain available for inspection and export.

For every coordinate \(j\), the diagnostic representation is

\[
z_{ij}=\frac{y_{ij}-\mu^{\mathrm{train}}_j}
              {\sigma^{\mathrm{train}}_j},
\qquad
\hat z_{ij}=\frac{\hat y_{ij}-\mu^{\mathrm{train}}_j}
                   {\sigma^{\mathrm{train}}_j}.
\]

The same complete-inner-training statistics are reused for training, validation, and test. Increase `INFERENCE_BATCH_SIZE` for throughput or reduce it if a batch does not fit on the selected GPU.

In [ ]:
def extract_predictions(model_output: Any) -> torch.Tensor:
    if isinstance(model_output, torch.Tensor):
        return model_output
    if (
        isinstance(model_output, (tuple, list))
        and model_output
        and isinstance(model_output[0], torch.Tensor)
    ):
        return model_output[0]
    raise TypeError(
        "Expected a prediction tensor or a tuple/list whose first item is one."
    )


def evaluate_split(
    split_name: str,
    indices: np.ndarray,
) -> dict[str, Any]:
    subset = Subset(dataset, indices.tolist())
    loader = ESMProteinDataLoader(
        subset,
        batch_size=INFERENCE_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda",
        max_sequence_length=dataset.max_sequence_length,
    )

    checkpoint_target_batches: list[torch.Tensor] = []
    checkpoint_prediction_batches: list[torch.Tensor] = []
    processed = 0

    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
    start_time = time.perf_counter()

    autocast_context = (
        torch.autocast(device_type="cuda", dtype=torch.bfloat16)
        if USE_BFLOAT16
        else nullcontext()
    )
    with torch.inference_mode(), autocast_context:
        for tokens, targets in loader:
            tokens = tokens.to(DEVICE, non_blocking=True)
            predictions = extract_predictions(model(tokens))
            checkpoint_target_batches.append(
                targets.detach().to(dtype=torch.float32, device="cpu")
            )
            checkpoint_prediction_batches.append(
                predictions.detach().to(dtype=torch.float32, device="cpu")
            )
            processed += targets.shape[0]
            print(
                f"[{split_name}] evaluated {processed:,}/{indices.size:,} proteins",
                end="\r",
            )

    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
    elapsed_seconds = time.perf_counter() - start_time
    print(
        f"[{split_name}] evaluated {processed:,}/{indices.size:,} proteins "
        f"in {elapsed_seconds:,.1f} s (batch size {INFERENCE_BATCH_SIZE:,})"
    )

    checkpoint_targets = torch.cat(checkpoint_target_batches)
    checkpoint_predictions = torch.cat(checkpoint_prediction_batches)
    if checkpoint_targets.shape != checkpoint_predictions.shape:
        raise ValueError(
            f"{split_name} teacher/prediction shape mismatch: "
            f"{tuple(checkpoint_targets.shape)} vs "
            f"{tuple(checkpoint_predictions.shape)}"
        )
    if checkpoint_targets.shape[0] != indices.size:
        raise ValueError(f"{split_name} output count does not match selected indices.")

    teacher = restore_target_coordinates(
        checkpoint_targets, target_mean, target_std, mode=target_normalization
    )
    predicted = restore_target_coordinates(
        checkpoint_predictions, target_mean, target_std, mode=target_normalization
    )
    teacher_standardized = transform_target_coordinates(
        teacher, target_mean, target_std, mode="standard"
    )
    predicted_standardized = transform_target_coordinates(
        predicted, target_mean, target_std, mode="standard"
    )

    median_predicted = median_baseline_vector.unsqueeze(0).expand_as(teacher).clone()
    median_predicted_standardized = (
        median_predicted - target_mean
    ) / target_std

    compared_tensors = (
        teacher,
        predicted,
        teacher_standardized,
        predicted_standardized,
        median_predicted,
        median_predicted_standardized,
    )
    if not all(torch.isfinite(values).all() for values in compared_tensors):
        raise ValueError(f"{split_name} contains non-finite embeddings.")

    standardized_residual = predicted_standardized - teacher_standardized
    median_standardized_residual = (
        median_predicted_standardized - teacher_standardized
    )
    standardized_cosine = F.cosine_similarity(
        teacher_standardized, predicted_standardized, dim=1
    )
    median_standardized_cosine = F.cosine_similarity(
        teacher_standardized, median_predicted_standardized, dim=1
    )

    records = [dataset.dataset[int(index)] for index in indices]
    per_protein = pd.DataFrame(
        {
            "split": split_name,
            "dataset_index": indices,
            "protein_id": [record["protein_id"] for record in records],
            "sequence_length": [len(record["sequence"]) for record in records],
            "standardized_cosine_similarity": standardized_cosine.numpy(),
            "median_standardized_cosine_similarity": (
                median_standardized_cosine.numpy()
            ),
            "mae_standardized": (
                standardized_residual.abs().mean(dim=1).numpy()
            ),
            "rmse_standardized": (
                standardized_residual.square().mean(dim=1).sqrt().numpy()
            ),
            "median_mae_standardized": (
                median_standardized_residual.abs().mean(dim=1).numpy()
            ),
            "median_rmse_standardized": (
                median_standardized_residual.square().mean(dim=1).sqrt().numpy()
            ),
        }
    )

    return {
        "indices": indices,
        "teacher": teacher.numpy(),
        "predicted": predicted.numpy(),
        "median_predicted": median_predicted.numpy(),
        "teacher_standardized": teacher_standardized.numpy(),
        "predicted_standardized": predicted_standardized.numpy(),
        "median_predicted_standardized": (
            median_predicted_standardized.numpy()
        ),
        "per_protein": per_protein,
        "elapsed_seconds": elapsed_seconds,
        "available_count": split_indices[split_name].size,
    }

In [ ]:
split_results = {
    split_name: evaluate_split(split_name, indices)
    for split_name, indices in selected_indices.items()
}
per_protein_metrics = pd.concat(
    [result["per_protein"] for result in split_results.values()],
    ignore_index=True,
)

if SAVE_RESULTS_TO is not None:
    output_path = Path(SAVE_RESULTS_TO).expanduser()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "checkpoint": str(CHECKPOINT_PATH),
            "checkpoint_epoch": checkpoint_epoch,
            "target_normalization": target_normalization or "none",
            "checkpoint_output_space": checkpoint_output_space,
            "target_statistics_path": str(target_statistics_path),
            "target_mean": target_mean,
            "target_std": target_std,
            "median_baseline_cache_path": str(median_baseline_cache_path),
            "median_baseline_cache_hit": median_baseline_cache_hit,
            "median_baseline_fit_indices": torch.tensor(
                inner_fold.train_idxs, dtype=torch.int64
            ),
            "median_baseline_vector": median_baseline_vector,
            "splits": {
                split_name: {
                    "dataset_indices": torch.from_numpy(result["indices"].copy()),
                    "protein_ids": result["per_protein"]["protein_id"].tolist(),
                    "teacher_embeddings": torch.from_numpy(result["teacher"]),
                    "predicted_embeddings": torch.from_numpy(result["predicted"]),
                    "median_baseline_embeddings": torch.from_numpy(
                        result["median_predicted"]
                    ),
                    "teacher_embeddings_standardized": torch.from_numpy(
                        result["teacher_standardized"]
                    ),
                    "predicted_embeddings_standardized": torch.from_numpy(
                        result["predicted_standardized"]
                    ),
                    "median_baseline_embeddings_standardized": torch.from_numpy(
                        result["median_predicted_standardized"]
                    ),
                    "per_protein_metrics": result["per_protein"].to_dict(
                        orient="list"
                    ),
                }
                for split_name, result in split_results.items()
            },
        },
        output_path,
    )
    print(f"Saved aligned split results to {output_path}")

## 1. Standardized cosine similarity

For each protein, standardized cosine similarity is

\[
\cos(z_i,\hat z_i)
=
\frac{z_i^\top\hat z_i}
     {\lVert z_i\rVert_2\lVert\hat z_i\rVert_2}.
\]

This removes the large shared coordinate-wise offset and rescales dimensions by their training-fold variability. It therefore asks whether the surrogate recovers the direction of the protein-specific deviation from the training mean.

The constant comparison here is the **coordinate-wise training median**. Its standardized vector is generally not zero because coordinate medians need not equal coordinate means:

\[
z_{\mathrm{median}}
=
\frac{\mathrm{median}_{\mathrm{train}}-\mu_{\mathrm{train}}}
     {\sigma_{\mathrm{train}}}.
\]

Consequently its cosine need not be zero, including on the training split. Moreover, averaging cosine normalizes every protein separately, so a zero average standardized embedding would still not imply zero average cosine. The plotted training subset may also differ from the complete fold used to fit the statistics. A training-**mean** baseline would become the zero vector after standardization; its cosine is mathematically undefined, so it is not used for this comparison.

In [ ]:
cosine_columns = {
    MODEL_LABEL: "standardized_cosine_similarity",
    MEDIAN_BASELINE_LABEL: "median_standardized_cosine_similarity",
}
predictor_colors = {
    MODEL_LABEL: "#4C72B0",
    MEDIAN_BASELINE_LABEL: MEDIAN_BASELINE_COLOR,
}

all_standardized_cosines = np.concatenate(
    [
        result["per_protein"][column].to_numpy()
        for result in split_results.values()
        for column in cosine_columns.values()
    ]
)
cosine_margin = max(
    0.04 * np.ptp(all_standardized_cosines),
    0.02,
)
cosine_limits = (
    max(-1.0, all_standardized_cosines.min() - cosine_margin),
    min(1.0, all_standardized_cosines.max() + cosine_margin),
)

fig, axes = plt.subplots(
    len(split_results),
    2,
    figsize=(13, 3.6 * len(split_results)),
    sharex="col",
)
if len(split_results) == 1:
    axes = np.asarray([axes])

for row, (split_name, result) in enumerate(split_results.items()):
    metrics = result["per_protein"]
    for predictor_name, column in cosine_columns.items():
        values = metrics[column].to_numpy()
        color = predictor_colors[predictor_name]
        axes[row, 0].hist(
            values,
            bins=HISTOGRAM_BINS,
            range=cosine_limits,
            density=True,
            histtype="step",
            linewidth=1.8,
            color=color,
            label=(
                f"{predictor_name}: median={np.median(values):.3f}, "
                f"5th pct={np.quantile(values, 0.05):.3f}"
            ),
        )
        sorted_values = np.sort(values)
        axes[row, 1].plot(
            sorted_values,
            np.arange(1, values.size + 1) / values.size,
            color=color,
            linewidth=1.8,
            label=predictor_name,
        )

    axes[row, 0].set(
        title=f"{split_name.title()}: standardized cosine distribution",
        ylabel="Density",
        xlim=cosine_limits,
    )
    axes[row, 1].set(
        title=f"{split_name.title()}: standardized cosine CDF",
        ylabel="Cumulative probability",
        xlim=cosine_limits,
        ylim=(0, 1),
    )
    axes[row, 0].legend(fontsize=9)
    axes[row, 1].legend(fontsize=9)

axes[-1, 0].set_xlabel("Standardized cosine similarity")
axes[-1, 1].set_xlabel("Standardized cosine similarity")
fig.tight_layout()
plt.show()

## 2. Reconstructed-vector \(R^2\) beyond the training-mean embedding

For a mean-centered checkpoint, the original-scale teacher and prediction are reconstructed as \(y_i=\mu_{\mathrm{train}}+c_i\) and \(\hat y_i=\mu_{\mathrm{train}}+\hat c_i\). The monitored score is computed directly on those reconstructed vectors:

\[
R^2_{\mathrm{reconstructed}}
=
1-
\frac{\sum_i \lVert \hat y_i-y_i\rVert_2^2}
     {\sum_i \lVert y_i-\mu_{\mathrm{train}}\rVert_2^2}.
\]

The denominator is exactly the squared error made by always predicting the coordinate-wise **training mean**. Thus:

- \(R^2=1\) is perfect prediction;
- \(R^2=0\) matches the training-mean baseline on that evaluated split;
- \(R^2<0\) is worse than predicting the training mean.

Because adding the same training mean to teacher and prediction leaves their residual unchanged, the numerator is exactly the summed squared error optimized by mean-centered MSE training. On a fixed evaluation set, this \(R^2\) therefore improves monotonically whenever that MSE decreases. The denominator deliberately uses the fixed training mean rather than re-centering on each evaluation split.

In [ ]:
def reconstructed_vector_r2_against_training_mean(
    teacher: np.ndarray,
    predicted: np.ndarray,
    training_mean: np.ndarray,
) -> float:
    residual_sum_of_squares = np.square(
        predicted - teacher
    ).sum(dtype=np.float64)
    mean_baseline_sum_of_squares = np.square(
        teacher - training_mean[None, :]
    ).sum(dtype=np.float64)
    if mean_baseline_sum_of_squares <= 0:
        return float("nan")
    return float(1.0 - residual_sum_of_squares / mean_baseline_sum_of_squares)


r2_values = {
    predictor_name: []
    for predictor_name in (MODEL_LABEL, MEDIAN_BASELINE_LABEL)
}
for result in split_results.values():
    r2_values[MODEL_LABEL].append(
        reconstructed_vector_r2_against_training_mean(
            result["teacher"],
            result["predicted"],
            target_mean.numpy(),
        )
    )
    r2_values[MEDIAN_BASELINE_LABEL].append(
        reconstructed_vector_r2_against_training_mean(
            result["teacher"],
            result["median_predicted"],
            target_mean.numpy(),
        )
    )

split_names = list(split_results)
x_positions = np.arange(len(split_names))
bar_width = 0.36
fig, axis = plt.subplots(figsize=(9, 4.8))

for offset, predictor_name in [
    (-bar_width / 2, MODEL_LABEL),
    (bar_width / 2, MEDIAN_BASELINE_LABEL),
]:
    bars = axis.bar(
        x_positions + offset,
        r2_values[predictor_name],
        width=bar_width,
        color=predictor_colors[predictor_name],
        alpha=0.88,
        label=predictor_name,
    )
    axis.bar_label(bars, fmt="%.3f", padding=3)

axis.axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
    label="Training-mean baseline",
)
axis.set(
    title=r"Reconstructed-vector variance recovered beyond the training mean",
    ylabel=r"Multivariate $R^2_{\mathrm{reconstructed}}$",
    xticks=x_positions,
    xticklabels=[name.title() for name in split_names],
)
axis.legend()
fig.tight_layout()
plt.show()

## 3. Standardized MAE and RMSE

For every protein, errors are computed after coordinate-wise training-fold standardization:

\[
\operatorname{MAE}_i
=
\frac{1}{d}\sum_j|\hat z_{ij}-z_{ij}|,
\qquad
\operatorname{RMSE}_i
=
\sqrt{\frac{1}{d}\sum_j(\hat z_{ij}-z_{ij})^2}.
\]

MAE reflects typical coordinate error and is comparatively robust to outliers. RMSE gives more weight to occasional large deviations. The box plots show the median, interquartile range, and 5th–95th percentile whiskers across proteins; extreme points are omitted from the drawing but remain part of the calculation.

In [ ]:
error_columns = {
    "Standardized MAE": {
        MODEL_LABEL: "mae_standardized",
        MEDIAN_BASELINE_LABEL: "median_mae_standardized",
    },
    "Standardized RMSE": {
        MODEL_LABEL: "rmse_standardized",
        MEDIAN_BASELINE_LABEL: "median_rmse_standardized",
    },
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
box_width = 0.28

for axis, (metric_name, columns) in zip(axes, error_columns.items()):
    for offset, predictor_name in [
        (-box_width / 1.7, MODEL_LABEL),
        (box_width / 1.7, MEDIAN_BASELINE_LABEL),
    ]:
        values_by_split = [
            split_results[split_name]["per_protein"][
                columns[predictor_name]
            ].to_numpy()
            for split_name in split_names
        ]
        boxplot = axis.boxplot(
            values_by_split,
            positions=x_positions + offset,
            widths=box_width,
            whis=(5, 95),
            showfliers=False,
            patch_artist=True,
            manage_ticks=False,
        )
        for box in boxplot["boxes"]:
            box.set(
                facecolor=predictor_colors[predictor_name],
                edgecolor=predictor_colors[predictor_name],
                alpha=0.65,
            )
        for median in boxplot["medians"]:
            median.set(color="black", linewidth=1.4)

    axis.set(
        title=f"Per-protein {metric_name}",
        ylabel=metric_name,
        xticks=x_positions,
        xticklabels=[name.title() for name in split_names],
    )

axes[0].legend(
    handles=[
        Patch(
            facecolor=predictor_colors[predictor_name],
            alpha=0.65,
            label=predictor_name,
        )
        for predictor_name in (MODEL_LABEL, MEDIAN_BASELINE_LABEL)
    ]
)
fig.tight_layout()
plt.show()

## 4. Preservation of pairwise distances

This diagnostic asks whether pairs of proteins that are close or far apart in teacher space retain the same ordering in surrogate space.

Within each split, a deterministic subset of at most `PAIRWISE_MAX_PROTEINS` is selected. Euclidean distances are computed between every pair in standardized space and divided by \(\sqrt{d}\) only to make the axes readable:

\[
d_{ik}
=
\frac{\lVert z_i-z_k\rVert_2}{\sqrt d}.
\]

The reported value is Spearman's rank correlation between teacher and surrogate pairwise distances. It is insensitive to monotonic rescaling but detects failures to preserve neighborhood ordering. The constant median baseline is omitted because all of its pairwise distances are zero, making rank correlation undefined.

In [ ]:
def pairwise_distance_diagnostics(
    result: dict[str, Any],
    *,
    seed: int,
) -> dict[str, Any]:
    teacher = result["teacher_standardized"]
    predicted = result["predicted_standardized"]
    sample_count = min(PAIRWISE_MAX_PROTEINS, teacher.shape[0])
    rng = np.random.default_rng(seed)
    sample_indices = np.sort(
        rng.choice(teacher.shape[0], size=sample_count, replace=False)
    )

    scale = np.sqrt(teacher.shape[1])
    teacher_distances = (
        pdist(teacher[sample_indices], metric="euclidean") / scale
    )
    predicted_distances = (
        pdist(predicted[sample_indices], metric="euclidean") / scale
    )
    correlation = float(
        spearmanr(teacher_distances, predicted_distances).statistic
    )

    plotted_pair_count = min(
        PAIRWISE_PLOT_MAX_PAIRS, teacher_distances.size
    )
    if plotted_pair_count < teacher_distances.size:
        plotted_pairs = rng.choice(
            teacher_distances.size,
            size=plotted_pair_count,
            replace=False,
        )
    else:
        plotted_pairs = np.arange(teacher_distances.size)

    return {
        "sample_count": sample_count,
        "pair_count": teacher_distances.size,
        "spearman_rho": correlation,
        "teacher_plot": teacher_distances[plotted_pairs],
        "predicted_plot": predicted_distances[plotted_pairs],
    }


pairwise_results = {
    split_name: pairwise_distance_diagnostics(
        result,
        seed=SUBSET_SEED + 1_000 + split_offset,
    )
    for split_offset, (split_name, result) in enumerate(split_results.items())
}

fig, axes = plt.subplots(
    1,
    len(pairwise_results),
    figsize=(5.2 * len(pairwise_results), 4.8),
)
if len(pairwise_results) == 1:
    axes = np.asarray([axes])

for axis, (split_name, diagnostics) in zip(axes, pairwise_results.items()):
    teacher_plot = diagnostics["teacher_plot"]
    predicted_plot = diagnostics["predicted_plot"]
    density = axis.hexbin(
        teacher_plot,
        predicted_plot,
        gridsize=60,
        bins="log",
        mincnt=1,
        cmap="viridis",
    )
    lower = min(teacher_plot.min(), predicted_plot.min())
    upper = max(teacher_plot.max(), predicted_plot.max())
    axis.plot(
        [lower, upper],
        [lower, upper],
        color="black",
        linestyle="--",
        linewidth=1,
    )
    axis.set(
        title=(
            f"{split_name.title()}: "
            rf"Spearman $\rho$={diagnostics['spearman_rho']:.3f}"
        ),
        xlabel=r"Teacher pairwise distance $/\sqrt{d}$",
        ylabel=r"Surrogate pairwise distance $/\sqrt{d}$",
        xlim=(lower, upper),
        ylim=(lower, upper),
    )
    fig.colorbar(density, ax=axis, label="log10(pair count)")
    axis.text(
        0.03,
        0.97,
        (
            f"{diagnostics['sample_count']:,} proteins\n"
            f"{diagnostics['pair_count']:,} pairs"
        ),
        transform=axis.transAxes,
        va="top",
        bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
    )

fig.tight_layout()
plt.show()

## 5. Original-scale L2 norm calibration

For each protein, this view compares the magnitude of the predicted embedding with the teacher embedding in the original foundation-model coordinate system:

\[
\lVert y_i\rVert_2
\quad\text{versus}\quad
\lVert\hat y_i\rVert_2.
\]

The dashed identity line represents perfect norm calibration. The legend reports \(R^2\) and mean absolute error between the two norms. These plots complement standardized cosine: cosine measures direction after removing the shared coordinate statistics, whereas this view tests whether the surrogate reproduces the original embedding magnitude. Matching norms alone does not imply that the individual coordinates are correct.

In [ ]:
def coefficient_of_determination(
    expected: np.ndarray,
    predicted: np.ndarray,
) -> float:
    residual_sum_of_squares = np.square(expected - predicted).sum(
        dtype=np.float64
    )
    total_sum_of_squares = np.square(expected - expected.mean()).sum(
        dtype=np.float64
    )
    if total_sum_of_squares <= 0:
        return float("nan")
    return float(1.0 - residual_sum_of_squares / total_sum_of_squares)


fig, axes = plt.subplots(
    1,
    len(split_results),
    figsize=(5.2 * len(split_results), 4.8),
)
if len(split_results) == 1:
    axes = np.asarray([axes])

for axis, (split_name, result) in zip(axes, split_results.items()):
    teacher_norm = np.linalg.norm(result["teacher"], axis=1)
    predicted_norm = np.linalg.norm(result["predicted"], axis=1)
    median_norm = np.linalg.norm(result["median_predicted"], axis=1)

    model_r2 = coefficient_of_determination(teacher_norm, predicted_norm)
    median_r2 = coefficient_of_determination(teacher_norm, median_norm)
    model_mae = np.abs(predicted_norm - teacher_norm).mean()
    median_mae = np.abs(median_norm - teacher_norm).mean()

    axis.scatter(
        teacher_norm,
        predicted_norm,
        color=predictor_colors[MODEL_LABEL],
        s=18,
        alpha=0.55,
        label=(
            f"{MODEL_LABEL}: "
            rf"$R^2$={model_r2:.3f}, MAE={model_mae:.3f}"
        ),
    )
    axis.scatter(
        teacher_norm,
        median_norm,
        color=predictor_colors[MEDIAN_BASELINE_LABEL],
        s=18,
        alpha=0.35,
        label=(
            f"{MEDIAN_BASELINE_LABEL}: "
            rf"$R^2$={median_r2:.3f}, MAE={median_mae:.3f}"
        ),
    )

    lower = min(teacher_norm.min(), predicted_norm.min(), median_norm.min())
    upper = max(teacher_norm.max(), predicted_norm.max(), median_norm.max())
    margin = max(0.04 * (upper - lower), 1e-6)
    limits = (lower - margin, upper + margin)
    axis.plot(
        limits,
        limits,
        color="black",
        linestyle="--",
        linewidth=1,
    )
    axis.set(
        title=f"{split_name.title()}: original-scale L2 norms",
        xlabel="Ground-truth L2 norm",
        ylabel="Predicted L2 norm",
        xlim=limits,
        ylim=limits,
    )
    axis.legend(fontsize=8)

fig.tight_layout()
plt.show()

## Reading the diagnostics together

- **Standardized cosine** measures the direction of each protein's deviation from the training mean.
- **Reconstructed-vector \(R^2_{\mathrm{reconstructed}}\)** measures original-scale vector fidelity relative to the training-mean predictor and is directly aligned with mean-centered MSE.
- **Standardized MAE/RMSE** measure coordinate fidelity and reveal the distribution of errors across proteins.
- **Pairwise-distance Spearman correlation** measures preservation of the global neighborhood ordering.
- **Original-scale L2 norm calibration** measures whether embedding magnitudes are reproduced.

Training-to-validation degradation indicates ordinary generalization loss; validation-to-test degradation indicates whether model-selection performance transfers to the untouched outer fold. A model can score well on one view and poorly on another—for example, correct directions but miscalibrated magnitudes—so the five views should be interpreted jointly.

All evaluation statistics are fitted without validation or test leakage. If the checkpoint changes during training, rerun model loading and every downstream cell together so all splits use the same snapshot.